In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("C:/Users/muham/Downloads/train.csv")

In [4]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [4]:
df = df.drop(columns=['PoolQC', 'MiscFeature', 'Alley', 'Fence'])

In [5]:
df['FireplaceQu'] = df['FireplaceQu'].fillna('None')

In [6]:
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())

In [7]:
categorical_cols = [cname for cname in df.columns if df[cname].dtype == "object"]
    
df = pd.get_dummies(df, columns=categorical_cols)

In [8]:
y = df['SalePrice']
X = df.drop(['SalePrice'], axis=1)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(f"Mean Absolute Error: ${mean_absolute_error(y_test, predictions):,.2f}")
print(f"This model has {r2_score(y_test, predictions) * 100:.2f}% accuracy")

Mean Absolute Error: $17,615.56
This model has 89.16% accuracy


In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(y_test, predictions, alpha=0.5, color='steelblue')

plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect prediction')

plt.xlabel('Actual Sale Price ($)')

plt.ylabel('Predicted Sale Price ($)')

plt.title('Actual vs Predicted House Prices')

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
test_df = pd.read_csv("C:/Users/muham/Downloads/test.csv")

test_df = test_df.drop(columns=['PoolQC', 'MiscFeature', 'Alley', 'Fence'], errors='ignore')
test_df['FireplaceQu'] = test_df['FireplaceQu'].fillna('None')
test_df['LotFrontage'] = test_df['LotFrontage'].fillna(test_df['LotFrontage'].median())

categorical_cols = [c for c in test_df.columns if test_df[c].dtype == 'object']
test_df = pd.get_dummies(test_df, columns=categorical_cols)

test_df = test_df.reindex(columns=X_train.columns, fill_value=0)

test_predictions = model.predict(test_df)

submission = pd.DataFrame({
    'Id': pd.read_csv("C:/Users/muham/Downloads/test.csv")['Id'],
    'SalePrice': test_predictions
})
submission.to_csv('submission.csv', index=False)